In [6]:
import os
import yaml
import requests

from pathlib import Path
from dataclasses import dataclass
from pymongo import MongoClient
from dotenv import load_dotenv
from box import ConfigBox

In [4]:
client = MongoClient(os.getenv("MONGODB_URI"))

db = client["kidney_db"]
collection = db["kidney_metadata"]

print("Connected Successfully")

Connected Successfully


In [5]:
documents = list(collection.find())

print("Total Images:", len(documents))

documents[0]

Total Images: 3842


{'_id': ObjectId('6a4a207645aa8616968ebafd'),
 'image_name': 'Normal- (1662).jpg',
 'label': 'Normal',
 'image_path': 'research/CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone\\Normal\\Normal- (1662).jpg'}

In [ ]:
def read_yaml(path_to_yaml: Path):
    with open(path_to_yaml) as yaml_file:
        return ConfigBox(yaml.safe_load(yaml_file))


def create_directories(paths):
    for path in paths:
        os.makedirs(path, exist_ok=True)

In [ ]:
config = read_yaml(Path("config/config.yaml"))
params = read_yaml(Path("params.yaml"))

load_dotenv()

MONGODB_URI = os.getenv("MONGODB_URI")

In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    database_name: str
    collection_name: str
    metadata_file: Path

In [ ]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories


In [ ]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            database_name=config.database_name,
            collection_name=config.collection_name,
            metadata_file=config.metadata_file
        )

        return data_ingestion_config

In [ ]:
import os
import shutil
from pymongo import MongoClient
from dotenv import load_dotenv

from cnnClassifier import logger
from cnnClassifier.entity.config_entity import DataIngestionConfig

load_dotenv()


class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def fetch_data_from_mongodb(self):

        logger.info("Connecting to MongoDB...")

        client = MongoClient(os.getenv("MONGODB_URI"))

        db = client[self.config.database_name]
        collection = db[self.config.collection_name]

        documents = collection.find()

        os.makedirs(self.config.root_dir, exist_ok=True)

        count = 0

        for doc in documents:

            image_path = os.path.normpath(doc["image_path"])
            label = doc["label"]

            destination_folder = os.path.join(self.config.root_dir, label)
            os.makedirs(destination_folder, exist_ok=True)

            destination_path = os.path.join(
                destination_folder,
                os.path.basename(image_path)
            )

            if os.path.exists(image_path):
                shutil.copy2(image_path, destination_path)
                count += 1
            else:
                logger.warning(f"Image not found: {image_path}")

        logger.info(f"Successfully copied {count} images.")

In [ ]:
try: 
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()

    data_ingestion = DataIngestion(config=data_ingestion_config)

        # Fetch metadata from MongoDB and recreate the dataset
    data_ingestion.fetch_data_from_mongodb()
except Exception as e:
    raise e    
